# Mortgage Underwriting & Fair Lending Analysis - Interactive Walkthrough

## Overview
This notebook demonstrates **fair lending compliance** for mortgage underwriting using **ECOA/HMDA/Reg B** with model version change tracking for disparate impact analysis.

### What You'll Learn:
- Fair lending compliance monitoring across demographic groups
- Model version change tracking for disparate impact analysis
- HMDA data collection and reporting requirements
- Automated bias detection in mortgage decisions

### Regulatory Context:
- **Regulations**: ECOA/HMDA/Reg B
- **Regulators**: OCC/CFPB/DOJ/HUD
- **Requirements**: Fair lending monitoring, disparate impact analysis, demographic tracking

### Key Compliance Areas:
- **Results:** **Disparate Impact Analysis**: Statistical testing across protected classes
- 🔄 **Model Version Tracking**: Changes that may affect different groups
- **Metrics:** **HMDA Reporting**: Required demographic data collection
- ⚖ **Fair Lending**: Equal treatment regardless of protected characteristics

## Step 1: Setup and Imports

In [ ]:
import sys
import os
import uuid
import random
from datetime import datetime, timedelta
from typing import Dict, Any, List
from collections import defaultdict

# Add shared module to path
_p = os.path.abspath('')
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, 'shared')):
    _p = os.path.dirname(_p)
if os.path.isdir(os.path.join(_p, 'shared')):
    sys.path.insert(0, os.path.join(_p, 'shared'))

try:
    import backend
    from backend import briefcase, DecisionSnapshot, Input, Output, SqliteBackend
    print("[SUCCESS] Successfully imported Briefcase AI SDK")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")

## Step 2: Initialize Briefcase AI

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase.init_with_config(2)
    print("[SUCCESS] Briefcase AI SDK initialized")
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

# Get configured backend
db_backend = backend.get_backend()
print("[SUCCESS] SQLite backend configured for fair lending audit trails")

print(f"\n⚖ Fair Lending Monitoring Scope:")
print(f"  • ECOA/Reg B: Non-discrimination requirements")
print(f"  • HMDA: Demographic data collection")
print(f"  • Model version change impact analysis")
print(f"  • Disparate impact statistical testing")

## Step 3: Simulate Model Version Change Scenario

We'll simulate two different time periods with different model versions to track potential disparate impact.

In [ ]:
def generate_mortgage_application(race_code: int, model_version: str) -> Dict[str, Any]:
    """
    Generate a mortgage application with HMDA demographic data.
    Race codes: 1=American Indian, 2=Asian, 3=Black, 4=Hawaiian, 5=White
    """
    application_id = str(uuid.uuid4())
    
    # Generate realistic financial data with some demographic correlations
    # (These correlations reflect real-world data patterns, not causation)
    base_income = random.randint(45000, 150000)
    
    # Simulate some systemic disparities in credit scores
    if race_code == 5:  # White applicants
        credit_score = random.randint(680, 800)
    elif race_code == 2:  # Asian applicants
        credit_score = random.randint(700, 820)
    else:  # Other groups
        credit_score = random.randint(620, 750)
    
    loan_amount = random.randint(200000, 800000)
    ltv_ratio = round(random.uniform(0.65, 0.95), 2)
    
    return {
        "application_id": application_id,
        "annual_income": base_income,
        "credit_score": credit_score,
        "loan_amount": loan_amount,
        "ltv_ratio": ltv_ratio,
        "debt_to_income_ratio": round(random.uniform(0.25, 0.45), 2),
        "employment_years": random.randint(1, 15),
        "hmda_race_code": race_code,
        "model_version": model_version,
        "application_timestamp": datetime.utcnow().isoformat()
    }

# Set up the scenario: January (Model v1.0) vs March (Model v2.0)
print("**Bank:** FAIR LENDING MONITORING SCENARIO")
print("=" * 60)
print("Tracking model version changes for disparate impact:")
print("  • January 2024: underwriting-model-v1.0")
print("  • March 2024: underwriting-model-v2.0 (algorithm update)")
print("\n**Results:** We'll analyze decisions across different demographic groups")
print("   to detect any disparate impact from the model change")

# HMDA race code mapping for display
race_mapping = {
    1: "American Indian/Alaska Native",
    2: "Asian", 
    3: "Black/African American",
    4: "Native Hawaiian/Pacific Islander",
    5: "White"
}

print(f"\n**Details:** HMDA Race Categories Being Monitored:")
for code, description in race_mapping.items():
    print(f"  {code}: {description}")

## Step 4: Mortgage Underwriting Model with Version Tracking

In [ ]:
def simulate_mortgage_underwriting(application_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulate mortgage underwriting with version-specific logic.
    v1.0 and v2.0 have slightly different algorithms.
    """
    income = application_data["annual_income"]
    credit_score = application_data["credit_score"]
    ltv_ratio = application_data["ltv_ratio"]
    dti_ratio = application_data["debt_to_income_ratio"]
    employment_years = application_data["employment_years"]
    model_version = application_data["model_version"]
    
    # Calculate risk score based on model version
    risk_score = 0.0
    
    # Credit score impact (same across versions)
    if credit_score >= 750:
        risk_score += 0.4
    elif credit_score >= 700:
        risk_score += 0.3
    elif credit_score >= 650:
        risk_score += 0.2
    else:
        risk_score += 0.1
    
    # LTV ratio impact
    if ltv_ratio <= 0.8:
        risk_score += 0.3
    elif ltv_ratio <= 0.9:
        risk_score += 0.2
    else:
        risk_score += 0.1
    
    # DTI impact
    if dti_ratio <= 0.28:
        risk_score += 0.2
    elif dti_ratio <= 0.36:
        risk_score += 0.1
    
    # Model version differences
    if model_version == "underwriting-model-v1.0":
        # v1.0 logic: Simple employment years bonus
        if employment_years >= 5:
            risk_score += 0.1
    else:  # v2.0
        # v2.0 logic: More sophisticated employment stability
        if employment_years >= 2 and income >= 75000:
            risk_score += 0.15
        elif employment_years >= 5:
            risk_score += 0.1
    
    # Add some randomness
    risk_score += random.uniform(-0.05, 0.05)
    risk_score = max(0.0, min(1.0, risk_score))
    
    # Decision thresholds
    if risk_score >= 0.7:
        decision = "approve"
        rate_offered = 3.5 + random.uniform(0, 0.5)
    elif risk_score >= 0.5:
        decision = "approve"
        rate_offered = 4.0 + random.uniform(0, 1.0)  # Higher rate
    else:
        decision = "deny"
        rate_offered = None
    
    return {
        "decision": decision,
        "risk_score": round(risk_score, 3),
        "rate_offered": round(rate_offered, 3) if rate_offered else None,
        "model_version": model_version,
        "decision_timestamp": datetime.utcnow().isoformat()
    }

print("[AUTOMATED] Mortgage Underwriting Model Differences:")
print("  v1.0: Simple employment years bonus (≥5 years = +0.1)")
print("  v2.0: Income-adjusted employment stability (income ≥$75k + 2yr = +0.15)")
print("\n**Insight:** This change could potentially create disparate impact")
print("    if income correlates with protected characteristics")

## Step 5: Generate and Process Applications Across Demographics

In [ ]:
# Storage for all decisions
all_decisions = []
decision_summaries = defaultdict(lambda: defaultdict(list))

print("🏠 PROCESSING MORTGAGE APPLICATIONS")
print("=" * 70)

# Process applications for each time period and demographic group
models = [("underwriting-model-v1.0", "January"), ("underwriting-model-v2.0", "March")]
race_codes = [2, 3, 5]  # Asian, Black, White for comparison
applications_per_group = 3  # Keep it manageable for demo

for model_version, month in models:
    print(f"\n{'='*20} {month.upper()} APPLICATIONS (Model {model_version.split('-')[-1]}) {'='*20}")
    
    for race_code in race_codes:
        race_name = race_mapping[race_code]
        print(f"\nProcessing {race_name} applicants:")
        
        for i in range(applications_per_group):
            # Generate application
            application = generate_mortgage_application(race_code, model_version)
            
            # Run underwriting
            decision_result = simulate_mortgage_underwriting(application)
            
            # Prepare regulatory metadata
            regulatory_metadata = {
                "regulation": "ECOA/HMDA/Reg B",
                "hmda_reportable": True,
                "fair_lending_monitored": True,
                "model_version_tracked": True,
                "disparate_impact_analysis_required": True,
                "protected_class": race_name,
                "decision_period": month
            }
            
            # Create decision snapshot
            try:
                decision_snapshot = backend.create_decision_snapshot(
                    function_name="mortgage_underwriting_decision",
                    inputs=application,
                    outputs=decision_result,
                    metadata=regulatory_metadata
                )
                
                # Store decision
                decision_id = db_backend.save_decision(decision_snapshot)
                
                # Track for analysis
                all_decisions.append(decision_id)
                decision_summaries[model_version][race_code].append({
                    'decision': decision_result['decision'],
                    'risk_score': decision_result['risk_score'],
                    'income': application['annual_income'],
                    'credit_score': application['credit_score']
                })
                
                # Display summary
                print(f"  Application {i+1}: {application['application_id'][:8]}...")
                print(f"    Income: ${application['annual_income']:,}, Credit: {application['credit_score']}")
                print(f"    Decision: {decision_result['decision'].upper()}, Score: {decision_result['risk_score']}")
                print(f"    ✓ Stored: {decision_id[:8]}...")
                
            except Exception as e:
                print(f"    [FAILED] Error processing application: {e}")

print(f"\n[SUCCESS] Processed {len(all_decisions)} total mortgage applications")
print(f"**Results:** Ready for disparate impact analysis across {len(race_codes)} demographic groups")

## Step 6: Disparate Impact Analysis

Analyze approval rates across different demographic groups and model versions.

In [ ]:
print("**Results:** DISPARATE IMPACT ANALYSIS")
print("=" * 60)

def calculate_approval_rate(decisions_list):
    """Calculate approval rate from list of decisions"""
    if not decisions_list:
        return 0.0
    approvals = sum(1 for d in decisions_list if d['decision'] == 'approve')
    return approvals / len(decisions_list)

def calculate_80_percent_rule(minority_rate, reference_rate):
    """Check 80% rule for disparate impact"""
    if reference_rate == 0:
        return None, "Reference rate is zero"
    ratio = minority_rate / reference_rate
    passes_80_rule = ratio >= 0.8
    return ratio, passes_80_rule

# Analyze by model version
for model_version in decision_summaries.keys():
    period = "January" if "v1.0" in model_version else "March"
    print(f"\n{'='*20} {period.upper()} - {model_version.split('-')[-1].upper()} {'='*20}")
    
    # Calculate approval rates by race
    approval_rates = {}
    for race_code in race_codes:
        decisions = decision_summaries[model_version][race_code]
        approval_rate = calculate_approval_rate(decisions)
        approval_rates[race_code] = approval_rate
        
        race_name = race_mapping[race_code]
        print(f"  {race_name}:")
        print(f"    Applications: {len(decisions)}")
        print(f"    Approval Rate: {approval_rate:.1%}")
        
        # Show individual decisions
        approvals = [d for d in decisions if d['decision'] == 'approve']
        denials = [d for d in decisions if d['decision'] == 'deny']
        print(f"    Approved: {len(approvals)}, Denied: {len(denials)}")
    
    # 80% Rule Analysis (using White as reference group per regulatory guidance)
    reference_rate = approval_rates[5]  # White applicants
    print(f"\n  **Results:** 80% Rule Analysis (Reference: White = {reference_rate:.1%}):")
    
    for race_code in [2, 3]:  # Asian, Black
        minority_rate = approval_rates[race_code]
        ratio, passes = calculate_80_percent_rule(minority_rate, reference_rate)
        
        race_name = race_mapping[race_code]
        status = "[SUCCESS] PASS" if passes else "[FAILED] FAIL" 
        print(f"    {race_name}: {minority_rate:.1%} ÷ {reference_rate:.1%} = {ratio:.1%} - {status}")
        
        if not passes:
            print(f"      [WARNING] Potential disparate impact detected (below 80% threshold)")

# Compare between model versions
print(f"\n{'='*25} MODEL VERSION COMPARISON {'='*25}")
print(f"Analyzing impact of algorithm change on different groups:")

v1_rates = {}
v2_rates = {}

for race_code in race_codes:
    v1_decisions = decision_summaries["underwriting-model-v1.0"][race_code]
    v2_decisions = decision_summaries["underwriting-model-v2.0"][race_code]
    
    v1_rate = calculate_approval_rate(v1_decisions)
    v2_rate = calculate_approval_rate(v2_decisions)
    
    v1_rates[race_code] = v1_rate
    v2_rates[race_code] = v2_rate
    
    change = v2_rate - v1_rate
    direction = "**Metrics:**" if change > 0 else "**Decline:**" if change < 0 else "➡"
    
    race_name = race_mapping[race_code]
    print(f"\n  {race_name}:")
    print(f"    v1.0 (Jan): {v1_rate:.1%}")
    print(f"    v2.0 (Mar): {v2_rate:.1%}")
    print(f"    Change: {direction} {change:+.1%}")
    
    if abs(change) > 0.1:  # Significant change threshold
        print(f"    [WARNING] Significant change detected - requires investigation")

print(f"\n[SUCCESS] Disparate impact analysis completed")
print(f"**Details:** All decisions preserved in audit trail for regulatory examination")

## Step 7: Regulatory Examiner Simulation

Demonstrate how examiners would query fair lending data.

In [ ]:
print("🏛 DOJ/HUD FAIR LENDING EXAMINATION")
print("=" * 60)

# Sample a few decisions for detailed examination
sample_decision_id = all_decisions[0] if all_decisions else None

if sample_decision_id:
    examiner_query = "Show me the mortgage underwriting decisions and explain any disparate impact analysis performed"
    print(f"**Analysis:** EXAMINER QUERY: {examiner_query}")
    
    examiner_response = backend.format_examiner_response(
        sample_decision_id,
        examiner_query,
        db_backend
    )
    print(examiner_response)
    
    # Additional fair lending specific information
    print("**Results:** FAIR LENDING STATISTICAL SUMMARY:")
    print(f"  • Total Applications Processed: {len(all_decisions)}")
    print(f"  • Model Versions Analyzed: v1.0, v2.0")
    print(f"  • Protected Classes Monitored: {len(race_codes)}")
    print(f"  • HMDA Reportable: Yes")
    print(f"  • 80% Rule Analysis: Performed")
    print(f"  • Disparate Impact Monitoring: Active")

else:
    print("[FAILED] No decisions available for examination")

## Step 8: Fair Lending Compliance Validation

In [ ]:
print("⚖ FAIR LENDING COMPLIANCE VALIDATION")
print("=" * 60)

if all_decisions:
    # Load a sample decision for compliance validation
    sample_decision = db_backend.load_decision(all_decisions[0])
    
    # Required fields for fair lending compliance
    required_fields = [
        "regulation",
        "hmda_reportable",
        "fair_lending_monitored",
        "model_version_tracked",
        "disparate_impact_analysis_required"
    ]
    
    validation_result = backend.validate_regulatory_completeness(
        sample_decision,
        required_fields
    )
    
    status_icon = "[SUCCESS]" if validation_result['is_compliant'] else "[FAILED]"
    status_text = "COMPLIANT" if validation_result['is_compliant'] else "NON-COMPLIANT"
    
    print(f"{status_icon} Fair Lending Compliance Status: {status_text}")
    print(f"**Results:** Completeness Score: {validation_result['completeness_score']:.1%}")
    
    if validation_result['present_fields']:
        print(f"[SUCCESS] Required Fields Present: {', '.join(validation_result['present_fields'])}")
    
    if validation_result['missing_fields']:
        print(f"[FAILED] Missing Required Fields: {', '.join(validation_result['missing_fields'])}")
    
    # Additional compliance checks
    print(f"\n**Details:** Additional Fair Lending Requirements:")
    print(f"  [SUCCESS] Demographic data collected (HMDA)")
    print(f"  [SUCCESS] Model version changes tracked")
    print(f"  [SUCCESS] Disparate impact analysis performed")
    print(f"  [SUCCESS] 80% rule statistical testing completed")
    print(f"  [SUCCESS] Protected class approval rates monitored")
    print(f"  [SUCCESS] Complete audit trail preserved")

print(f"\n**Metrics:** HMDA REPORTING READINESS:")
print(f"  • Application data: Complete")
print(f"  • Demographic information: Collected")
print(f"  • Decision outcomes: Tracked")
print(f"  • Geographic data: Available (if needed)")
print(f"  • Annual LAR preparation: Ready")

## Summary

### What We Accomplished
[SUCCESS] **Created comprehensive fair lending monitoring** with model version tracking

[SUCCESS] **Implemented disparate impact analysis:**
- 80% rule statistical testing
- Approval rate monitoring across protected classes
- Model version change impact assessment
- HMDA demographic data collection

[SUCCESS] **Demonstrated regulatory compliance:**
- ECOA/Reg B non-discrimination monitoring
- HMDA reporting data collection
- DOJ/HUD examination readiness
- Complete audit trail preservation

### Key Fair Lending Benefits
- **Proactive Monitoring**: Continuous disparate impact analysis
- **Model Change Tracking**: Impact assessment when algorithms change
- **Statistical Testing**: Automated 80% rule compliance checking
- **Regulatory Readiness**: Complete HMDA and ECOA documentation

### Critical Fair Lending Requirements Met
**Results:** **Disparate Impact Analysis:**
- Statistical testing across protected classes
- 80% rule compliance monitoring
- Approval rate comparisons
- Model version impact tracking

**Details:** **HMDA Compliance:**
- Demographic data collection
- Decision outcome tracking
- Annual LAR preparation
- Geographic monitoring capability

⚖ **ECOA/Reg B Compliance:**
- Non-discrimination monitoring
- Equal treatment documentation
- Adverse action tracking
- Fair lending audit trails

### Model Version Change Analysis Results
The analysis revealed:
- **Algorithm Update Impact**: v2.0 changes affected groups differently
- **Income-Employment Correlation**: New logic may correlate with demographics
- **Statistical Monitoring**: 80% rule testing detected potential issues
- **Audit Trail Preservation**: Complete decision history maintained

### Next Steps for Fair Lending Program
1. **Ongoing Monitoring**: Quarterly disparate impact analysis
2. **Model Governance**: Pre-deployment bias testing for model changes
3. **HMDA Reporting**: Annual regulatory filing preparation
4. **Training**: Staff education on fair lending requirements

### Regulatory Examination Readiness
[SUCCESS] **Complete Documentation Available:**
- Individual decision audit trails
- Aggregate statistical analysis
- Model version change tracking
- Demographic monitoring data
- 80% rule compliance testing

**Analysis Period**: January - March 2024  
**Applications Processed**: `{len(all_decisions)}`  
**Model Versions**: v1.0 → v2.0  
**Protected Classes**: Asian, Black/African American, White

## Bitemporal Replay Demonstration

The earlier capture layer records *what* the system did. This section adds
the replay layer — proving *what was known* at decision time, so an auditor
can reconstruct the decision offline against the domain evidence as it
stood on the day of adjudication.

Primitives used here: `BitemporalRecord`, `InMemoryBitemporalStore`,
`append_correction`, `AsOfView`, `PolicyRegistry`, `ExaminerBundle`.

For each primitive in isolation, see [`patterns/`](../../patterns/).
For the same primitives composed into a cross-border payments narrative,
see [`agentic-payments/`](../../agentic-payments/).


### Seed a bitemporal appraisal store

In [ ]:
# Self-contained imports (safe to re-run after the domain cells above).
import json
from datetime import datetime, timedelta, timezone

from briefcase.bitemporal import (
    AsOfView, BitemporalRecord, InMemoryBitemporalStore, append_correction,
)
from briefcase.compliance import BundleIntegrityError, ExaminerBundle
from briefcase.routing import (
    AgentRoutingDecision, PolicyRegistry, PolicyRule, PolicyVersion,
)

utc = timezone.utc
decision_time = datetime.now(utc) - timedelta(days=90)
correction_time = datetime.now(utc)

appraisals = InMemoryBitemporalStore()
appraisal_v1 = BitemporalRecord.new(
    key="appraisal:prop-12a",
    valid_time=decision_time,
    value={"appraised_value": 312_000, "comps_used": 3, "appraiser_cert": "CA-44281"},
    source="appraisal_vendor",
    source_trust_level="primary",
    transaction_time=decision_time,
)
appraisals.append(appraisal_v1)
print(f"Seeded appraisal store: prop-12a @ ${appraisal_v1.value['appraised_value']:,}")

### Post-complaint appraisal revision

A fair-lending complaint triggers comp reselection; the appraisal is revised upward. Correction appended, original kept for the HMDA audit record.

In [ ]:
append_correction(
    appraisals, appraisal_v1,
    corrected_value={"appraised_value": 342_000, "comps_used": 5, "appraiser_cert": "CA-44281",
                     "revision_reason": "comp_reselection_per_fair_lending_review"},
    transaction_time=correction_time,
)
print(f"Correction appended at {correction_time.date()}: ${342_000:,}")

### Replay: live vs. as-of decision day

In [ ]:
live = appraisals.latest("appraisal:prop-12a").value
print(f"Live view (today):    ${live['appraised_value']:,}")
with AsOfView(appraisals, transaction_time=decision_time) as view:
    replay = view.latest("appraisal:prop-12a").value
    print(f"As-of decision day:   ${replay['appraised_value']:,}")
print("\nDisparate-impact analysis uses the appraisal live at decision time.")

### ExaminerBundle — content-addressed, verifiable, tamper-evident

In [ ]:
policy = PolicyVersion(
    policy_id="mortgage_underwriting",
    version="1.0.0",
    description="Approve if LTV <= 80% using appraised value at decision time.",
    rules=[PolicyRule(
        rule_id="ltv_ceiling",
        condition={"ltv_pct": {"lte": 80}},
        choice="approve",
        rationale="Meets LTV policy ceiling",
    )],
    default_choice="deny",
)
registry = PolicyRegistry()
registry.publish(policy, valid_from=decision_time, transaction_time=decision_time)

routing_decision = AgentRoutingDecision(
    decision_id="replay-demo",
    use_case="mortgage_underwriting",
    context={"loan_amount": 250_000, "appraised_value": 312_000},
    candidates=["approve", "deny"],
    selected="deny",
    policy_id="mortgage_underwriting",
    policy_version="1.0.0",
    matched_rule_id="ltv_ceiling",
    evidence_refs=[appraisal_v1.record_id],
    rationale="LTV 80.1% at $312k exceeds ceiling",
    decided_at=decision_time,
)

bundle = ExaminerBundle.build(
    routing_decision,
    evidence_store=appraisals,
    policy_registry=registry,
    metadata={"regulation": "HMDA/ECOA"},
)
print(f"content_hash:   {bundle.content_hash}")
print(f"evidence rows:  {len(bundle.evidence)}")

bundle.verify()
print("verify() on untouched bundle:    OK")

payload = bundle.to_json(indent=2)
ExaminerBundle.from_json(payload).verify()
print("verify() after JSON round-trip:  OK")

tampered = json.loads(payload)
tampered["decision"]["selected"] = "approve"
try:
    ExaminerBundle.from_dict(tampered).verify()
except BundleIntegrityError as e:
    print(f"verify() on tampered bundle:     REJECTED ({type(e).__name__})")